# ISOM 835 · Explainer — Logistic Regression, Plain and Simple
**Suffolk University · Sawyer Business School · Fall 2026 · Prof. Hasan Arslan**

**The one idea:** logistic regression looks at a customer and answers **"what is the chance this customer leaves?"** with a number between 0% and 100%.

That's it. No formulas tonight. Run each cell (Shift + Enter) and read the short note above it.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

**Setup: just run this cell.** It loads the Telco customers from Session 1 and defines two small chart helpers. You don't need to read it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression

df = pd.read_csv('https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv')
df['left'] = (df['Churn'] == 'Yes').astype(int)                                # 1 = cancelled, 0 = stayed
df['month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)        # 1 = no long-term contract
months = pd.DataFrame({'tenure': range(0, 121)})

def show_dots():
    jitter = np.random.default_rng(835).uniform(-0.04, 0.04, len(df))
    plt.figure(figsize=(8, 3.6))
    plt.scatter(df['tenure'], df['left'] + jitter, s=4, alpha=0.15, color='gray')
    plt.axhline(0, color='black', lw=0.6); plt.axhline(1, color='black', lw=0.6)
    plt.xlabel('tenure (months with the company)'); plt.ylabel('left? (0 = no, 1 = yes)')

def finish(title):
    plt.title(title); plt.legend(); plt.show()

print(f'{len(df):,} customers, {df["left"].mean():.0%} of them left')

## Step 1. Why not a straight line?

We want to predict **left (1) or stayed (0)** from **tenure** (months with the company). First, try a straight line.

**Look at the right end of the red line.**

In [ ]:
line = LinearRegression().fit(df[['tenure']], df['left'])

show_dots()
plt.plot(months['tenure'], line.predict(months), color='#d64545', lw=2, label='straight line')
finish('A straight line goes below 0')

The line drops **below 0**. A chance below 0% is impossible. We need a line that **bends** and stays between 0 and 1.

## Step 2. The S-curve

That bending line is **logistic regression**. Same data, same one fact (tenure):

In [ ]:
model = LogisticRegression().fit(df[['tenure']], df['left'])

show_dots()
plt.plot(months['tenure'], model.predict_proba(months)[:, 1], color='#2e8b57', lw=3, label='logistic regression')
finish('Logistic regression bends and stays between 0 and 1')

**Read it like a manager:** new customers are the most likely to leave, and the risk drops the longer they stay.

## Step 3. Ask the model about any customer

`predict_proba` means "what's the chance?" Give it a few tenures and it answers:

In [ ]:
customers = pd.DataFrame({'tenure': [1, 12, 36, 72]})
customers['chance of leaving'] = model.predict_proba(customers)[:, 1].round(2)
customers

## Step 4. Add a second fact

Customers on a **month-to-month** plan can leave any time. Let's tell the model that too: `month_to_month` is 1 for those customers and 0 for everyone else.

Then we ask about **two customers with the same tenure**, one on month-to-month and one with a contract.

In [ ]:
model2 = LogisticRegression().fit(df[['tenure', 'month_to_month']], df['left'])

two = pd.DataFrame({'tenure': [12, 12], 'month_to_month': [1, 0]})
two['chance of leaving'] = model2.predict_proba(two)[:, 1].round(2)
two

Same tenure, very different risk. **More facts help the model tell customers apart.**

## Step 5. From chance to yes/no

A chance isn't a decision. `predict` turns each chance into **yes/no** with a cutoff of **0.5**: above 50% means "will leave."

The next cell counts how many customers the model flags at 0.5, and how many of the customers who really left it catches.

In [ ]:
df['chance'] = model2.predict_proba(df[['tenure', 'month_to_month']])[:, 1]
df['flagged'] = (df['chance'] >= 0.5).astype(int)

caught = ((df['flagged'] == 1) & (df['left'] == 1)).sum()
print(f'customers flagged at 0.5:   {df["flagged"].sum():,}')
print(f'leavers caught:             {caught:,} of the {df["left"].sum():,} who left')

**Surprise:** at 0.5 the model catches only about **1 in 5** of the customers who left. Most risky customers sit between 20% and 50%, so they never cross the line. The model *does* know who is risky; the **0.5 line is in the wrong place**. Let's try lower lines.

In [ ]:
for cutoff in [0.5, 0.3, 0.2]:
    flagged = df['chance'] >= cutoff
    caught = (flagged & (df['left'] == 1)).sum()
    print(f'cutoff {cutoff}: flag {flagged.sum():5,} customers → {caught:5,} of the {df["left"].sum():,} who left are caught')

**Lower cutoff → we catch more leavers, and we bother more customers who would have stayed.** Choosing the cutoff is a **business** decision about what each mistake costs. That's Session 5.

## Your turn

1. In Step 3, ask about customers with **6, 24, and 48** months. *Hint: change the numbers in the list.*
2. In Step 4, compare month-to-month vs. contract at **3 months** instead of 12. Is the gap bigger or smaller?
3. In two sentences for your manager, with no code words: what does logistic regression give us, and why not a straight line?

In [ ]:
# Your turn — work here

## What we learned

- **Logistic regression gives a chance between 0% and 100%.**
- **A straight line doesn't work** for yes/no: it goes below 0 and above 1.
- **`predict_proba`** = "what's the chance?" **`predict`** = "yes or no?" using a 0.5 cutoff.
- **More facts** (like contract type) help the model separate risky customers from safe ones.
- **The cutoff is a business choice**, coming up in Session 5.